# Bike Sharing — Exploratory Data Analysis (EDA)

**Author:** Miki  

**Goal of the project:** predict the *daily* number of bike rentals (`cnt`) from weather and calendar information, using linear regression.

**`cnt` is the label (target)** we want to predict. Every other column is a possible input.

In this notebook I just *look* at the data: its shape, quality, the target, and how each feature relates to `cnt`. No modelling here — that happens later in `src/train.py`.

## 1. Setup and load the data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# make the plots a bit nicer
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

In [ ]:
# the notebook lives in notebooks/, the data lives in datasets/raw/
df = pd.read_csv('../datasets/raw/day.csv')
df.head()

## 2. Basic structure of the dataset

How many rows and columns do we have, and what type is each column?

In [ ]:
df.shape  # (rows, columns)

In [ ]:
df.info()

## 3. Summary statistics

`describe()` gives min/max/mean/quartiles for the numeric columns. This is a quick way to spot strange values.

In [ ]:
df.describe()

## 4. Data quality checks

Before trusting the data we check for **missing values** and **duplicate rows**.

In [ ]:
# missing values per column
df.isnull().sum()

In [ ]:
# number of duplicated rows
df.duplicated().sum()

## 5. The target variable: `cnt`

`cnt` = total bikes rented that day (it is the sum of `casual` + `registered`).
Let's look at its distribution.

In [ ]:
df['cnt'].describe()

In [ ]:
plt.figure(figsize=(8,4))
sns.histplot(df['cnt'], bins=30, kde=True)
plt.title('Distribution of daily bike rentals (cnt)')
plt.xlabel('cnt (rentals per day)')
plt.ylabel('number of days')
plt.show()

## 6. Columns we will DROP (and why)

Following the team contract we drop these 4 columns before modelling:

- `instant` — just a row id, no predictive value.
- `dteday` — the calendar date; the useful parts of it (`yr`, `mnth`, `weekday`) are already separate columns.
- `casual` and `registered` — these two **add up exactly to `cnt`**, so using them would be *cheating* (data leakage): the model would just sum them. We prove that below.

In [ ]:
# casual + registered should equal cnt for every row
leak_check = (df['casual'] + df['registered'] == df['cnt']).all()
print('casual + registered == cnt for all rows:', leak_check)

## 7. Features we will USE

These are the 11 input features agreed in the contract. I split them into *categorical* (codes/categories) and *numeric* (continuous numbers) just for the plots below.

In [ ]:
target = 'cnt'

# categorical-style features (stored as numbers, but they represent categories)
categorical = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']

# continuous numeric features
numeric = ['temp', 'atemp', 'hum', 'windspeed']

features = categorical + numeric
print('Target  :', target)
print('Features:', features)

## 8. Correlation heatmap

How strongly is each feature linearly related to `cnt` (and to each other)? Values close to +1 / -1 mean a strong linear relationship — exactly what linear regression can use.

In [ ]:
plt.figure(figsize=(10,8))
corr = df[features + [target]].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation between features and cnt')
plt.show()

In [ ]:
# just the correlation of each feature with the target, sorted
corr[target].drop(target).sort_values(ascending=False)

## 9. Numeric features vs `cnt`

Scatter plots show the relationship between each continuous feature and the number of rentals.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12,8))
for ax, col in zip(axes.flat, numeric):
    sns.scatterplot(data=df, x=col, y=target, alpha=0.4, ax=ax)
    ax.set_title(f'{col} vs cnt')
plt.tight_layout()
plt.show()

## 10. Categorical features vs `cnt`

Box plots show how the rentals differ across the categories (e.g. season, weather situation).

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14,11))
for ax, col in zip(axes.flat, categorical):
    sns.boxplot(data=df, x=col, y=target, ax=ax)
    ax.set_title(f'{col} vs cnt')
# hide the 2 empty subplots (we have 7 features, 9 slots)
for ax in axes.flat[len(categorical):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# average rentals per month, to see the seasonal pattern clearly
df.groupby('mnth')[target].mean().plot(kind='bar', figsize=(9,4), title='Average cnt per month')
plt.ylabel('average cnt')
plt.show()

## 11. Key takeaways

*(Fill these in after you run the notebook — write what YOU see, in your own words. Some things to look for:)*

- Is the data clean (no missing values / duplicates)?
- Which features correlate most with `cnt`? (temperature usually does.)
- Do `temp` and `atemp` look almost identical? (They are highly correlated — worth mentioning.)
- How does `cnt` change with season, weather situation, and year?

These observations justify the modelling choices made later in `src/train.py`.